In [1]:
import os
import json
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc  # Memory Management

# ==============================================================================
# 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# ==============================================================================
path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
with open(path_mappa, "r") as f:
    mappa_strategie = json.load(f)

tier1_keys = [k for k in mappa_strategie.keys() if "tier 1" in k.lower() or "tier_1" in k.lower()]
tier2_keys = [k for k in mappa_strategie.keys() if "tier 2" in k.lower() or "tier_2" in k.lower()]

ticker_target = []
for k in tier1_keys + tier2_keys:
    ticker_target.extend(mappa_strategie[k])

BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_dict = {}

if 'dati_puliti' in locals() or 'dati_puliti' in globals():
    for cluster, tickers in dati_puliti.items():
        for t, df in tickers.items():
            if t in ticker_target:
                all_tickers_dict[t] = {"df": df, "cluster": cluster}
else:
    print("📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...")
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        t = file.replace(".parquet", "")
                        if t in ticker_target:
                            all_tickers_dict[t] = {
                                "df": pd.read_parquet(os.path.join(cluster_path, file)),
                                "cluster": cluster
                            }

print(f"🚀 Motore Donchian Asimmetrico 2D (Pure Price Action) su {len(all_tickers_dict)} asset.")

# ==============================================================================
# 2. GENERAZIONE RANGE PARAMETRI REGOLARIZZATI (DONCHIAN 2D)
# ==============================================================================
upper_range = np.arange(10, 85, 5)   # Finestra Breakout Ingresso (Massimi)
lower_range = np.arange(5, 45, 5)    # Finestra Trailing Stop Uscita (Minimi)

# ==============================================================================
# 3. CORE FUNCTION: SPLIT 60/40 + CONVOLUZIONE 2D DONCHIAN CHANNELS
# ==============================================================================

def calc_advanced_metrics(rets, giorni_anno):
    """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    return cagr, vol, sharpe, sortino, max_dd


def ottimizza_donchian_2d(ticker, df, upper_range, lower_range, is_crypto=False):
    close = df['Close'].replace(0, np.nan).bfill().ffill().copy()
    high = df['High'].replace(0, np.nan).bfill().ffill().copy()
    low = df['Low'].replace(0, np.nan).bfill().ffill().copy()
    
    close.index = pd.to_datetime(close.index)
    
    giorni_anno = 365 if is_crypto else 252
    total_bars = len(close)
    split_idx = int(total_bars * 0.6)
    
    if split_idx < 200 or (total_bars - split_idx) < 50:
        return None
        
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    anni_is = len(close_is) / giorni_anno
    
    # --- CALCOLO BUY & HOLD ---
    rets_bh_is = close_is.pct_change().dropna()
    cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
    rets_bh_oos = close_oos.pct_change().dropna()
    cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)

    # ==============================================================
    # 1. COSTRUZIONE COMBINAZIONI MATRICIALI ED ESTRAZIONE SEZNALI IS
    # ==============================================================
    griglia_2d = [(u, l) for u in upper_range for l in lower_range if u > l]
    
    unique_u = sorted(list(set([c[0] for c in griglia_2d])))
    unique_l = sorted(list(set([c[1] for c in griglia_2d])))
    
    hh_dict = {u: high.iloc[:split_idx].rolling(u).max().shift(1) for u in unique_u}
    ll_dict = {l: low.iloc[:split_idx].rolling(l).min().shift(1) for l in unique_l}
    
    entries_list, exits_list = [], []
    for u, l in griglia_2d:
        entries_list.append(close_is.vbt.crossed_above(hh_dict[u]))
        exits_list.append(close_is.vbt.crossed_below(ll_dict[l]))
        
    entries_df = pd.concat(entries_list, axis=1)
    exits_df = pd.concat(exits_list, axis=1)
    
    pf_is = vbt.Portfolio.from_signals(close_is, entries_df, exits_df, freq='D')
    
    std_is = pf_is.returns().std()
    tpy_is = pf_is.trades.count() / anni_is if anni_is > 0 else 0
    sharpes_is = pf_is.sharpe_ratio()
    
    valid_sharpes = np.where((std_is > 0) & (tpy_is >= 2), sharpes_is, np.nan)
    
    if np.isnan(valid_sharpes).all():
        del entries_df, exits_df, pf_is
        gc.collect()
        return "LOW_SHARPE"
        
    # ==============================================================
    # 2. MAPPATURA MATRICE E FASE DI CONVOLUZIONE 2D PURA
    # ==============================================================
    shape_2d = (len(upper_range), len(lower_range))
    mat_sharpe = np.full(shape_2d, np.nan)
    
    for idx, (u, l) in enumerate(griglia_2d):
        u_idx = np.where(upper_range == u)[0][0]
        l_idx = np.where(lower_range == l)[0][0]
        mat_sharpe[u_idx, l_idx] = valid_sharpes[idx]
        
    convolved_candidates = []
    
    for u_idx in range(shape_2d[0]):
        for l_idx in range(shape_2d[1]):
            if np.isnan(mat_sharpe[u_idx, l_idx]):
                continue
                
            u_min, u_max = max(0, u_idx-1), min(shape_2d[0], u_idx+2)
            l_min, l_max = max(0, l_idx-1), min(shape_2d[1], l_idx+2)
            
            vicinato = mat_sharpe[u_min:u_max, l_min:l_max].flatten()
            valid_neighbors = vicinato[~np.isnan(vicinato)]
            
            if len(valid_neighbors) >= 4:
                conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
                convolved_candidates.append((conv_score, (upper_range[u_idx], lower_range[l_idx])))
                
    convolved_candidates.sort(key=lambda x: x[0], reverse=True)
    
    del entries_df, exits_df, pf_is
    gc.collect()
    
    if not convolved_candidates:
        return "LOW_SHARPE"
        
    best_u, best_l = convolved_candidates[0][1]
        
    # ==============================================================
    # 3. BACKTEST OUT-OF-SAMPLE SULLA COPPIA OTTIMALE (TOP 1)
    # ==============================================================
    final_hh = high.rolling(best_u).max().shift(1)
    final_ll = low.rolling(best_l).min().shift(1)
    
    final_entries = close.vbt.crossed_above(final_hh).vbt.signals.fshift(1)
    final_exits = close.vbt.crossed_below(final_ll).vbt.signals.fshift(1)
    
    pf_i = vbt.Portfolio.from_signals(close_is, final_entries.iloc[:split_idx], final_exits.iloc[:split_idx], freq='D')
    is_rets = pf_i.returns()
    trd_is = pf_i.trades.count()
    win_rate_is = ((pf_i.trades.pnl > 0).sum() / trd_is * 100) if trd_is > 0 else 0
    
    pf_o = vbt.Portfolio.from_signals(close_oos, final_entries.iloc[split_idx:], final_exits.iloc[split_idx:], freq='D')
    oos_rets = pf_o.returns()
    trd_oos = pf_o.trades.count()
    win_rate_oos = ((pf_o.trades.pnl > 0).sum() / trd_oos * 100) if trd_oos > 0 else 0
        
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(is_rets, giorni_anno)
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(oos_rets, giorni_anno)
    
    alpha_oos = cagr_oos - cagr_bh_oos
    param_str = f"({int(best_u)},{int(best_l)})"
    
    del pf_i, pf_o
    gc.collect()

    # --- 🔥 INSERITI METADATI RAW COMPLETI PER EXPORT SPECCHIATO ---
    return {
        "Parametri": param_str,
        "Parametri_Raw": [[int(best_u), int(best_l)]],
        "Giorni_Anno": giorni_anno,
        "Inizio_OOS": data_inizio_oos,
        
        # Strategy Metrics
        "SHP_IS": shp_is, "SHP_OOS": shp_oos,
        "CAGR_IS": cagr_is, "CAGR_OOS": cagr_oos,
        "DD_IS": dd_is, "DD_OOS": dd_oos,
        "VOL_IS": vol_is, "VOL_OOS": vol_oos,
        "SORT_IS": sort_is, "SORT_OOS": sort_oos,
        "TRD_IS": trd_is, "TRD_OOS": trd_oos,
        "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
        
        # Benchmark Metrics (Buy & Hold)
        "BH_SHP_IS": shp_bh_is, "BH_SHP_OOS": shp_bh_oos,
        "BH_CAGR_IS": cagr_bh_is, "BH_CAGR_OOS": cagr_bh_oos,
        "BH_DD_IS": dd_bh_is, "BH_DD_OOS": dd_bh_oos,
        "BH_VOL_IS": vol_bh_is, "BH_VOL_OOS": vol_bh_oos,
        "BH_SORT_IS": sort_bh_is, "BH_SORT_OOS": sort_bh_oos,
        
        "Alpha_OOS": alpha_oos
    }

# ==============================================================================
# 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# ==============================================================================
report_finale = []

for ticker, info in all_tickers_dict.items():
    df = info["df"]
    cluster_appartenenza = info["cluster"]
    is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
    print(f"\n============================================================")
    print(f"⌛ ELABORAZIONE DONCHIAN 2D {ticker:<8} | Cluster: {cluster_appartenenza}")
    
    metrics = ottimizza_donchian_2d(ticker, df, upper_range, lower_range, is_crypto=is_crypto)
    
    if isinstance(metrics, dict):
        print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
        print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
        print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
        print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
        print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
        print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
        metrics["Ticker"] = ticker
        metrics["Cluster"] = cluster_appartenenza
        report_finale.append(metrics)
    elif metrics == "LOW_SHARPE":
        print("⚠️ SCARTATO: Nessun altopiano solido trovato In-Sample.")

# ==============================================================================
# 5. LEADERBOARD FINALE
# ==============================================================================
if len(report_finale) > 0:
    df_leaderboard = pd.DataFrame(report_finale)
    df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

    print("\n" + "="*160)
    print("🏆 TABELLONE FINALE DONCHIAN CONVOLUZIONE 2D (Ordinato per Alpha OOS)")
    print("="*160)

    tabella_stampa = df_leaderboard[[
        "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
        "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
        "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
    ]].copy()

    tabella_stampa.columns = [
        "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Up-Low)", 
        "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
        "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
    ]

    print(tabella_stampa.to_string(
        index=False, justify='left', col_space=9,
        formatters={
            'SHP IS':    '{:,.2f}'.format, 'B&H IS':    '{:,.2f}'.format,
            'SHP OOS':   '{:,.2f}'.format, 'B&H OOS':   '{:,.2f}'.format,
            'CAGR OOS':  lambda x: f"{x*100:,.1f}%", 'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
            'ALPHA %':   lambda x: f"{x*100:,.1f}%", 'DD OOS':    lambda x: f"{x*100:,.1f}%",
            'B&H DD':    lambda x: f"{x*100:,.1f}%", 'TRD OOS':   '{:,.1f}'.format
        }
    ))
    print("="*160)
else:
    print("\n⚠️ Nessun asset ha superato i filtri operativi.")

# ==============================================================================
# 🔥 6. SALVATAGGIO CONFIGURAZIONE JSON PER INTERFACCIA ENSEMBLE
# ==============================================================================
if len(report_finale) > 0:
    export_dir = "progetto_trading_data"
    os.makedirs(export_dir, exist_ok=True)
    path_export = os.path.join(export_dir, "config_donchian.json")
    
    config_json = {
        "NOME_STRATEGIA": "Donchian_Channels_2D_Convolution",
        "DATA_GENERAZIONE": "2026-06-21",
        "ASSETS": {}
    }
    
    for item in report_finale:
        ticker = item["Ticker"]
        config_json["ASSETS"][ticker] = {
            "Cluster": item["Cluster"],
            "Data_Inizio_OOS": item["Inizio_OOS"],
            "Giorni_Anno": int(item["Giorni_Anno"]),
            "Numero_Modelli_Ensemble": len(item["Parametri_Raw"]),
            "Parametri_Ottimi": item["Parametri_Raw"],
            "Alpha_OOS": float(item["Alpha_OOS"]),
            
            "Metrics_In_Sample": {
                "Sharpe_IS": float(item["SHP_IS"]),
                "Sortino_IS": float(item["SORT_IS"]),
                "CAGR_IS": float(item["CAGR_IS"]),
                "Volatilità_IS": float(item["VOL_IS"]),
                "MaxDD_IS": float(item["DD_IS"]),
                "Trades_IS": float(item["TRD_IS"]),
                "WinRate_IS": float(item["WIN_IS"])
            },
            "Metrics_Out_Of_Sample": {
                "Sharpe_OOS": float(item["SHP_OOS"]),
                "Sortino_OOS": float(item["SORT_OOS"]),
                "CAGR_OOS": float(item["CAGR_OOS"]),
                "Volatilità_OOS": float(item["VOL_OOS"]),
                "MaxDD_OOS": float(item["DD_OOS"]),
                "Trades_OOS": float(item["TRD_OOS"]),
                "WinRate_OOS": float(item["WIN_OOS"])
            },
            "Benchmark_In_Sample": {
                "Sharpe_BH_IS": float(item["BH_SHP_IS"]),
                "Sortino_BH_IS": float(item["BH_SORT_IS"]),
                "CAGR_BH_IS": float(item["BH_CAGR_IS"]),
                "Volatilità_BH_IS": float(item["BH_VOL_IS"]),
                "MaxDD_BH_IS": float(item["BH_DD_IS"])
            },
            "Benchmark_Out_Of_Sample": {
                "Sharpe_BH_OOS": float(item["BH_SHP_OOS"]),
                "Sortino_BH_OOS": float(item["BH_SORT_OOS"]),
                "CAGR_BH_OOS": float(item["BH_CAGR_OOS"]),
                "Volatilità_BH_OOS": float(item["BH_VOL_OOS"]),
                "MaxDD_BH_OOS": float(item["BH_DD_OOS"])
            }
        }
        
    with open(path_export, "w") as f:
        json.dump(config_json, f, indent=4)
        
    print(f"\n💾 Configurazione globale Donchian salvata con successo in: {path_export} 🚀")

📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...
🚀 Motore Donchian Asimmetrico 2D (Pure Price Action) su 90 asset.

⌛ ELABORAZIONE DONCHIAN 2D COCOA.c  | Cluster: cash_cfd
✅ Parametri Scelti: (70,5) | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.72 | Cagr: 59.6% | DD: -25.2% | Sort: 1.23 | Trd: 7.0 | Win: 71.4%
  [IS]  B&H   -> Shp: 1.67 | Cagr: 102.3% | DD: -45.7% | Sort: 2.00
  [OOS] STRAT -> Shp: -2.22 | Cagr: -20.3% | DD: -26.6% | Sort: -0.77 | Trd: 2.0 | Win: 0.0%
  [OOS] B&H   -> Shp: -1.21 | Cagr: -54.0% | DD: -73.9% | Sort: -1.86
  🔥 ALPHA OOS:  33.72%

⌛ ELABORAZIONE DONCHIAN 2D COFFEE.c | Cluster: cash_cfd
✅ Parametri Scelti: (35,20) | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 0.58 | Cagr: 11.4% | DD: -24.9% | Sort: 0.61 | Trd: 5.0 | Win: 60.0%
  [IS]  B&H   -> Shp: 1.50 | Cagr: 53.4% | DD: -25.4% | Sort: 2.53
  [OOS] STRAT -> Shp: -0.18 | Cagr: -5.3% | DD: -19.7% | Sort: -0.10 | Trd: 2.0 | Win: 50.0%
  [OOS] B&H   -> Shp: -0